# Puyo Puyo AI — Self-Play 強化学習（Google Colab GPU）

このノートブックは Google Colab の GPU（T4 / A100）を使って
`puyo-trainer` の `self-play` バイナリを実行し、
学習済みモデルを GitHub にプッシュします。

## 前提条件

1. **GPU ランタイムを選択してください**  
   メニュー → `ランタイム` → `ランタイムのタイプを変更` → `T4 GPU`（または A100）

2. **Colab シークレットに `GITHUB_TOKEN` を設定**  
   左サイドバー 🔑 → `GITHUB_TOKEN` を追加  
   GitHub の Fine-grained PAT、権限: `Contents: Read and write`

## セッション切れ後の再開手順

Colab のセッションは約 90 分（無料）でリセットされます。  
再接続後は以下のセルを順に実行してください：

| セル | 毎回必要か | 補足 |
|------|-----------|------|
| 3: GPU 確認 | ✅ 推奨 | スキップ可 |
| 4: Rust インストール | ✅ 必要 | セッション揮発 |
| 5: リポジトリ | ✅ 必要 | clone/pull + モデル昇格を自動実行 |
| 7: ビルド | ✅ 必要 | ビルドキャッシュも揮発 |
| 8: self-play 実行 | ✅ 必要 | 完了後に GitHub へ自動プッシュ |

## 継続学習の流れ

セッションをまたいでも、毎回セル 3 → 8 を順番に実行するだけです。  
セル 5 が前回の `puyo_model_selfplay.bin` を自動的に `puyo_model.bin` に昇格させるため、  
前回の self-play 結果を起点にして学習が続きます。


In [ ]:
# ============================================================
# セル 3: GPU / CUDA 環境確認
# ============================================================
import subprocess
import glob
import os

print('=== GPU 確認 ===')
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('GPU が見つかりません。ランタイムを GPU に変更してください。')
    raise SystemExit('GPU ランタイムが必要です')

print('\n=== CUDA ライブラリ確認 ===')
cuda_libs = (
    glob.glob('/usr/local/cuda*/lib64/libcuda.so*') +
    glob.glob('/usr/lib/x86_64-linux-gnu/libcuda.so*')
)
print('libcuda:', cuda_libs if cuda_libs else '見つかりません')

# burn/cuda-jit が参照する CUDA_PATH を設定
cuda_dirs = sorted(glob.glob('/usr/local/cuda*'))
CUDA_PATH = cuda_dirs[-1] if cuda_dirs else '/usr/local/cuda'

os.environ['CUDA_PATH'] = CUDA_PATH
os.environ['CUDA_HOME'] = CUDA_PATH

print(f'\nCUDA_PATH={CUDA_PATH}')
print('nvcc（コンパイラ）は burn/cuda-jit の JIT 実行には不要です')

In [ ]:
# ============================================================
# セル 4: Rust インストール
# ============================================================
import subprocess
import os
import glob

# rustup でインストール
result = subprocess.run(
    'curl --proto =https --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable',
    shell=True, capture_output=False
)

# PATH を永続化
cargo_bin = '/root/.cargo/bin'
os.environ['PATH'] = f"{cargo_bin}:{os.environ['PATH']}"

# CUDA_PATH も再設定（セルをスキップして実行された場合に備えて）
cuda_dirs = sorted(glob.glob('/usr/local/cuda*'))
CUDA_PATH = cuda_dirs[-1] if cuda_dirs else '/usr/local/cuda'
os.environ['CUDA_PATH'] = CUDA_PATH
os.environ['CUDA_HOME'] = CUDA_PATH

# バージョン確認
for cmd in [['rustc', '--version'], ['cargo', '--version']]:
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout.strip() if r.returncode == 0 else f'{cmd[0]} が見つかりません')

In [ ]:
# ============================================================
# セル 5: リポジトリ clone / pull → self-play モデル昇格
# Colab 左サイドバー 🔑 に GITHUB_TOKEN（Fine-grained PAT）を設定してください
# 必要な権限: Contents (read/write)
# ============================================================
import subprocess
import shutil
import os
from google.colab import userdata

REPO_DIR = '/content/puyopuyo-ai'

# PAT でクローン（push 時にも認証に使用）
token = userdata.get('GITHUB_TOKEN')
REPO_URL = f'https://{token}@github.com/hfappmaker/puyopuyo-ai.git'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    print('リポジトリ既存 → git pull')
    subprocess.run(['git', 'pull'], cwd=REPO_DIR, text=True)
else:
    print('リポジトリ clone 中...')
    result = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], text=True)
    if result.returncode != 0:
        raise RuntimeError('clone 失敗。GITHUB_TOKEN の権限を確認してください。')
    print('clone 完了')

# artifacts ディレクトリを確認
artifacts_dir = os.path.join(REPO_DIR, 'artifacts')
os.makedirs(artifacts_dir, exist_ok=True)
print('\nartifacts/ の内容:')
for f in sorted(os.listdir(artifacts_dir)):
    size = os.path.getsize(os.path.join(artifacts_dir, f))
    print(f'  {f}: {size:,} bytes')

# ============================================================
# self-play モデルをベースモデルに昇格（継続学習用）
# puyo_model_selfplay.bin が存在する場合、puyo_model.bin として使用します。
# 初回（selfplay.bin がない）はスキップされ、元の puyo_model.bin がそのまま使われます。
# ============================================================
src = os.path.join(REPO_DIR, 'artifacts/puyo_model_selfplay.bin')
dst = os.path.join(REPO_DIR, 'artifacts/puyo_model.bin')

print('\n=== モデル昇格チェック ===')
if os.path.exists(src):
    # 初回昇格時のみ元モデルをバックアップ
    bak = dst + '.orig_backup'
    if os.path.exists(dst) and not os.path.exists(bak):
        shutil.copy2(dst, bak)
        print('元モデルをバックアップ: puyo_model.bin.orig_backup')
    shutil.copy2(src, dst)
    size = os.path.getsize(dst)
    print(f'昇格完了: puyo_model_selfplay.bin → puyo_model.bin ({size:,} bytes)')
    print('次回の self-play はこのモデルを起点に学習します。')
else:
    print('puyo_model_selfplay.bin が見つかりません → 初回実行: 元の puyo_model.bin を使用します。')

In [ ]:
# ============================================================
# セル 7: ビルド（GPU）
# 初回は 10〜20 分かかります。Colab の出力に進捗が表示されます。
# ============================================================
import subprocess
import os
import glob

REPO_DIR = '/content/puyopuyo-ai'

# 環境変数を再確認
os.environ['PATH'] = f"/root/.cargo/bin:{os.environ.get('PATH', '')}"
cuda_dirs = sorted(glob.glob('/usr/local/cuda*'))
CUDA_PATH = cuda_dirs[-1] if cuda_dirs else '/usr/local/cuda'
os.environ['CUDA_PATH'] = CUDA_PATH
os.environ['CUDA_HOME'] = CUDA_PATH

print(f'CUDA_PATH={CUDA_PATH}')
print('GPU ビルド開始（初回は時間がかかります）...')

result = subprocess.run(
    ['cargo', 'build', '--release', '-p', 'puyo-trainer', '--bin', 'self-play'],
    cwd=REPO_DIR,
    env=os.environ,
)

if result.returncode == 0:
    binary = os.path.join(REPO_DIR, 'target/release/self-play')
    size = os.path.getsize(binary)
    print(f'\nビルド成功: {binary} ({size:,} bytes)')
else:
    print(f'\nGPU ビルド失敗 (returncode={result.returncode})')
    print('→ 下のフォールバックセルで CPU ビルドを試みてください')

In [ ]:
# ============================================================
# セル 8: self-play 実行 → GitHub 自動プッシュ
# リアルタイムでログが流れます。5000 ゲーム完了まで待ちます。
# 完了後は自動的に GitHub にプッシュします。
# ============================================================
import subprocess
import os
import time
import glob
from google.colab import userdata

REPO_DIR = '/content/puyopuyo-ai'
binary = os.path.join(REPO_DIR, 'target/release/self-play')

if not os.path.exists(binary):
    raise FileNotFoundError('バイナリが見つかりません。セル 7 のビルドを先に実行してください。')

# CUDA_PATH を再設定（JIT コンパイル時に参照される）
cuda_dirs = sorted(glob.glob('/usr/local/cuda*'))
CUDA_PATH = cuda_dirs[-1] if cuda_dirs else '/usr/local/cuda'
os.environ['CUDA_PATH'] = CUDA_PATH
os.environ['CUDA_HOME'] = CUDA_PATH

print('self-play 開始')
print('設定: NUM_GAMES=5000, LEARNING_RATE=1e-4, BATCH_SIZE=512')
print('注意: CUDA JIT の初回コンパイル（最初の数ゲーム）はやや遅いです')
print('-' * 60)

start_time = time.time()
output_lines = []

# cwd をリポジトリルートに設定（artifacts/ への相対パス参照のため必須）
process = subprocess.Popen(
    [binary],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ,
)

for line in process.stdout:
    print(line, end='', flush=True)
    output_lines.append(line)

process.wait()
elapsed = time.time() - start_time

print('-' * 60)
if process.returncode != 0:
    print(f'self-play 失敗 (returncode={process.returncode})')
    print('最後の 15 行:')
    for line in output_lines[-15:]:
        print(line, end='')
else:
    print(f'self-play 完了！経過時間: {elapsed / 60:.1f} 分')

    # ============================================================
    # GitHub にプッシュ
    # ============================================================
    token = userdata.get('GITHUB_TOKEN')
    remote_url = f'https://{token}@github.com/hfappmaker/puyopuyo-ai.git'

    subprocess.run(['git', 'config', 'user.email', 'colab-training@example.com'], cwd=REPO_DIR)
    subprocess.run(['git', 'config', 'user.name', 'Colab Training'], cwd=REPO_DIR)
    subprocess.run(['git', 'remote', 'set-url', 'origin', remote_url], cwd=REPO_DIR)

    files_to_commit = [
        'artifacts/puyo_model_selfplay.bin',
        'artifacts/puyo_model.bin',
        'artifacts/norm_params.txt',
    ]
    staged = []
    for f in files_to_commit:
        path = os.path.join(REPO_DIR, f)
        if os.path.exists(path):
            subprocess.run(['git', 'add', f], cwd=REPO_DIR)
            staged.append(f)

    if not staged:
        print('\nコミットするファイルがありません')
    else:
        timestamp = time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())
        result = subprocess.run(
            ['git', 'commit', '-m', f'chore: update self-play model [{timestamp}]'],
            cwd=REPO_DIR, capture_output=True, text=True
        )
        print(f'\n{result.stdout.strip()}')
        if result.returncode != 0:
            print('コミット失敗:', result.stderr.strip())
        else:
            print('プッシュ中...')
            result = subprocess.run(
                ['git', 'push', 'origin', 'main'],
                cwd=REPO_DIR, capture_output=True, text=True
            )
            if result.returncode == 0:
                print('プッシュ完了！')
            else:
                print('プッシュ失敗:')
                print(result.stderr.strip())
                print('→ GITHUB_TOKEN の Contents write 権限を確認してください')

## 完了！

セル 8 が正常終了すると、モデルが自動的に GitHub にプッシュされます。

### GitHub にプッシュされるファイル

| ファイル | 内容 |
|---------|------|
| `artifacts/puyo_model_selfplay.bin` | self-play で更新されたモデル |
| `artifacts/puyo_model.bin` | 昇格済みモデル（次回の起点） |
| `artifacts/norm_params.txt` | 正規化パラメータ |

### 継続学習の流れ

セッションをまたいで再帰的に学習を継続する場合は、毎回セル 1 → 8 を順番に実行するだけです。  
セル 5 が `puyo_model_selfplay.bin` を自動的に `puyo_model.bin` に昇格させるため、  
前回の self-play 結果を起点にして学習が続きます。

### 次のステップ（ローカルで作業する場合）

1. `git pull` でモデルを取得
2. WASM を再ビルド:
   ```bash
   bash scripts/build-wasm.sh
   ```
